<a href="https://colab.research.google.com/github/priyaaaww/ITA06_LAB_EXP/blob/main/EXP_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import csv
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# STEP 1: FILE PATH
# Upload data.csv to Google Colab's Files panel first.
FILE_PATH = "/content/breast-cancer_dataset"

if not os.path.exists(FILE_PATH):
    from google.colab import files
    print("Please upload your data.csv file:")
    uploaded = files.upload()
    FILE_PATH = next(iter(uploaded))

print("Loading file:", FILE_PATH)

# STEP 2: READ AND FIX CSV
with open(FILE_PATH, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.reader(f))

if len(rows) < 2:
    raise ValueError("CSV file is empty or contains no data rows.")

header = rows[0]

# Remove extra empty columns from the header
while header and not header[-1].strip():
    header.pop()

data = []
for row in rows[1:]:
    if not row or not any(x.strip() for x in row):
        continue

    # Remove extra fields and pad missing fields
    row = row[:len(header)]
    row += [""] * (len(header) - len(row))
    data.append(row)

df = pd.DataFrame(data, columns=header)
df.columns = df.columns.str.strip()

# STEP 3: CLEAN DATA
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df = df.drop(columns=["id"], errors="ignore")

if "diagnosis" not in df.columns:
    raise ValueError(
        "Target column 'diagnosis' not found. "
        "Check the CSV header and select the correct target column."
    )

df["diagnosis"] = (
    df["diagnosis"].astype(str).str.strip().str.upper()
)
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

X = df.drop(columns=["diagnosis"]).apply(
    pd.to_numeric, errors="coerce"
)
X = X.replace([np.inf, -np.inf], np.nan)

valid = X.notna().all(axis=1) & df["diagnosis"].notna()
X = X.loc[valid].to_numpy(dtype=float)
y = df.loc[valid, "diagnosis"].to_numpy(dtype=int)

if len(X) == 0 or len(np.unique(y)) != 2:
    raise ValueError("Not enough valid data for binary classification.")

print("Dataset loaded:", X.shape)

# STEP 4: SPLIT DATA
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

y_train = y_train.reshape(-1, 1).astype(float)

# STEP 5: NORMALIZE FEATURES
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# STEP 6: INITIALIZE ANN
np.random.seed(42)

input_size = X_train.shape[1]
hidden_size = 16
learning_rate = 0.05
epochs = 1000

W1 = np.random.randn(input_size, hidden_size) * 0.1
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, 1) * 0.1
b2 = np.zeros((1, 1))

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# STEP 7: TRAIN USING BACKPROPAGATION
for epoch in range(epochs):
    # Forward propagation
    hidden = sigmoid(X_train @ W1 + b1)
    output = sigmoid(hidden @ W2 + b2)

    # Backpropagation
    error_output = output - y_train

    dW2 = hidden.T @ error_output / len(X_train)
    db2 = np.mean(error_output, axis=0, keepdims=True)

    error_hidden = (
        error_output @ W2.T
    ) * hidden * (1 - hidden)

    dW1 = X_train.T @ error_hidden / len(X_train)
    db1 = np.mean(error_hidden, axis=0, keepdims=True)

    # Update weights and biases
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if (epoch + 1) % 200 == 0:
        h = sigmoid(X_train @ W1 + b1)
        o = sigmoid(h @ W2 + b2)

        loss = -np.mean(
            y_train * np.log(np.clip(o, 1e-10, 1)) +
            (1 - y_train) * np.log(np.clip(1 - o, 1e-10, 1))
        )
        print(f"Epoch {epoch + 1}, Loss: {loss:.4f}")

# STEP 8: TEST ANN
hidden_test = sigmoid(X_test @ W1 + b1)
output_test = sigmoid(hidden_test @ W2 + b2)
y_pred = (output_test >= 0.5).astype(int).ravel()

# STEP 9: DISPLAY RESULTS
print("\nANN USING BACKPROPAGATION")
print("-------------------------")
print("Input neurons:", input_size)
print("Hidden neurons:", hidden_size)
print("Output neurons: 1")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Benign", "Malignant"],
    zero_division=0
))

Loading file: /content/breast-cancer_dataset
Dataset loaded: (569, 30)
Epoch 200, Loss: 0.2837
Epoch 400, Loss: 0.1528
Epoch 600, Loss: 0.1135
Epoch 800, Loss: 0.0956
Epoch 1000, Loss: 0.0855

ANN USING BACKPROPAGATION
-------------------------
Input neurons: 30
Hidden neurons: 16
Output neurons: 1
Training samples: 455
Testing samples: 114
Accuracy: 98.25%

Confusion Matrix:
[[72  0]
 [ 2 40]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.97      1.00      0.99        72
   Malignant       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

